In [ ]:
# Cell 1 — Imports, load, drop leaky columns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

df = pd.read_csv('../data/raw/hotel_bookings.csv')

# agent and company are numeric IDs stored as float (because of NaN).
# Cast to string now so OneHotEncoder sees uniform string input later.
df['agent'] = df['agent'].fillna('Unknown').astype(str)
df['company'] = df['company'].fillna('Unknown').astype(str)

# Drop leaky columns — these reveal the answer at prediction time
# reservation_status: 'Canceled' / 'Check-Out' / 'No-Show' — directly encodes is_canceled
# reservation_status_date: date the status was set — same problem
LEAKY = ['reservation_status', 'reservation_status_date']
df = df.drop(columns=LEAKY)

X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('Class balance:\n', y.value_counts(normalize=True))

In [2]:
# Cell 2 — Train / test split (80 / 20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,   # makes the split reproducible — same split every run
    stratify=y         # keeps 63/37 class balance in both sets
)

print('Train size:', X_train.shape[0])
print('Test size: ', X_test.shape[0])
print('Train cancellation rate:', y_train.mean().round(3))
print('Test cancellation rate: ', y_test.mean().round(3))

Train size: 95512
Test size:  23878
Train cancellation rate: 0.37
Test cancellation rate:  0.37


In [3]:
# Cell 3 — Define feature lists + build ColumnTransformer
#
# ColumnTransformer applies DIFFERENT preprocessing to DIFFERENT column types:
#   Numeric cols  -> fill missing with median, then scale to mean=0 std=1
#   Categorical cols -> fill missing with 'Unknown', then one-hot encode
#
# Everything is fit ONLY on training data (inside the Pipeline later).
# The test set never influences the imputer or scaler.

numeric_cols = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes',
    'days_in_waiting_list', 'adr',
    'required_car_parking_spaces', 'total_of_special_requests'
]

categorical_cols = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel',
    'reserved_room_type', 'assigned_room_type',
    'deposit_type', 'agent', 'company', 'customer_type'
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print('Preprocessor built — numeric cols:', len(numeric_cols), '| categorical cols:', len(categorical_cols))

Preprocessor built — numeric cols: 17 | categorical cols: 12


In [4]:
# Cell 4 — Build the full Pipeline, fit, evaluate
#
# Pipeline = preprocessor + model chained together.
# pipeline.fit(X_train, y_train) runs the preprocessor on training data ONLY,
# then trains the model on the transformed features.
# pipeline.predict(X_test) applies the SAME transformations (no refitting) to test data.

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(max_iter=1000, random_state=42))
])

print('Fitting... (may take 30-60 seconds)')
pipeline.fit(X_train, y_train)
print('Done!')

y_pred = pipeline.predict(X_test)

f1 = f1_score(y_test, y_pred)
print(f'\nBaseline Logistic Regression F1 score: {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Not cancelled', 'Cancelled']))

Fitting... (may take 30-60 seconds)


TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['float', 'str']